# LLM Evaluation

We compare two RAG prompts against the ground-truth question set (`ground-truth-generation.ipynb`),
using an LLM-as-judge to classify each generated answer's relevance to its question.
The winning prompt is wired into `rag.py`'s `prompt_template`.


In [1]:
import pandas as pd
df_question = pd.read_csv('../data/ground-truth-retrieval.csv')
df_question.head()

,chunk_id,question
0,604727_0,What is the origin of coffee consumption accor...
1,604727_0,Which two main types of coffee beans are menti...
2,604727_0,How has the global coffee industry economicall...
3,604727_0,What are some common ways coffee can be prepar...
4,604727_0,Where are coffee plants primarily cultivated a...


## Candidate prompts

`rag.prompt_template` (the current production default) vs. `prompt_template_v2`, which
explicitly allows the model to say "I don't know" when the context doesn't cover the question.


In [2]:
prompt_template_v2 = """
You're a coffee assistant. Answer the QUESTION based on the CONTEXT from our coffee database.
Use only the facts from the CONTEXT. If the CONTEXT doesn't contain enough information to answer, say "I don't know" instead of guessing.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [3]:
import sys
sys.path.append('..')

from coffee_assistant import rag

In [4]:
df_sample = df_question.sample(n=100, random_state=1)
sample = df_sample.to_dict(orient='records')

## LLM-as-judge

`prompt2_template` asks gpt-4o-mini to classify a (question, generated answer) pair as
`NON_RELEVANT`, `PARTLY_RELEVANT`, or `RELEVANT`, returned as parsable JSON.


In [5]:
prompt2_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

## Evaluation pipeline

`evaluate_answers` generates an answer per sampled question with a given prompt template,
then judges it; `to_dataframe` flattens the results into a plain table.


In [6]:
from tqdm.auto import tqdm
import json 

def evaluate_answers(sample, prompt_template):
    evaluations = []

    for record in tqdm(sample):
        question = record['question']
        answer_llm = rag.rag(question, prompt_template=prompt_template)
    
        prompt = prompt2_template.format(
            question=question,
            answer_llm=answer_llm['answer']
        )
    
        evaluation, _ = rag.llm(prompt)
        evaluation = json.loads(evaluation)
        
        evaluations.append((record, answer_llm, evaluation))
    return evaluations


In [7]:
def to_dataframe(evaluations):
    df_eval = pd.DataFrame(evaluations, columns=['record', 'answer', 'evaluation'])
    
    df_eval['chunk_id'] = df_eval.record.apply(lambda d: d['chunk_id'])
    df_eval['question'] = df_eval.record.apply(lambda d: d['question'])
    
    df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
    df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])
    
    del df_eval['record']
    del df_eval['evaluation']
    
    return df_eval

### Run both prompts

100 sampled questions per prompt, ~200 LLM calls total (100 generation + 100 judging) per prompt.


In [8]:
evaluations_default = evaluate_answers(sample, rag.prompt_template)
df_default = to_dataframe(evaluations_default)

evaluations_v2 = evaluate_answers(sample, prompt_template_v2)
df_v2 = to_dataframe(evaluations_v2)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

## Relevance distribution


In [9]:
df_default.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.97
PARTLY_RELEVANT    0.02
NON_RELEVANT       0.01
Name: proportion, dtype: float64

In [10]:
df_v2.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.91
NON_RELEVANT       0.06
PARTLY_RELEVANT    0.03
Name: proportion, dtype: float64

### Inspecting non-relevant cases

Spot-check the few answers each prompt's judge marked `NON_RELEVANT`.


In [12]:
df_default[df_default.relevance == 'NON_RELEVANT']

,answer,chunk_id,question,relevance,explanation
95,{'answer': 'The provided context does not cont...,4604645_28,What is the typical serving size and compositi...,NON_RELEVANT,The generated answer does not provide any info...


In [13]:
df_v2[df_v2.relevance == 'NON_RELEVANT']

,answer,chunk_id,question,relevance,explanation
1,"{'answer': 'I don't know.', 'model_used': 'gpt...",19619306_13,What two types of coffee are combined in the p...,NON_RELEVANT,The generated answer 'I don't know' does not p...
37,"{'answer': 'I don't know.', 'model_used': 'gpt...",19619306_14,What are some alternative names for drinks tha...,NON_RELEVANT,The generated answer 'I don't know' does not p...
38,"{'answer': 'I don't know.', 'model_used': 'gpt...",604727_41,What was the percentage increase in coffee con...,NON_RELEVANT,The generated answer 'I don't know' does not p...
52,"{'answer': 'I don't know.', 'model_used': 'gpt...",604727_40,What brewing method remains the most popular f...,NON_RELEVANT,The generated answer 'I don't know' does not p...
55,"{'answer': 'I don't know.', 'model_used': 'gpt...",604727_21,How did coffee cultivation practices change st...,NON_RELEVANT,The generated answer 'I don't know' does not p...
95,"{'answer': 'I don't know', 'model_used': 'gpt-...",4604645_28,What is the typical serving size and compositi...,NON_RELEVANT,The generated answer 'I don't know' does not p...


### Save results


In [16]:
df_default.to_csv('../data/rag-eval-default.csv', index=False)
df_v2.to_csv('../data/rag-eval-v2.csv', index=False)

## Results

Compared the production prompt (`rag.prompt_template`) against a variant that
explicitly allows saying "I don't know" when the context is insufficient, judging
100 sampled questions with gpt-4o-mini as an LLM judge (NON_RELEVANT / PARTLY_RELEVANT / RELEVANT).

| Prompt | Relevant | Partly relevant | Non relevant |
|---|---|---|---|
| **Default** | **0.97** | 0.02 | 0.01 |
| "I don't know" variant | 0.91 | 0.03 | 0.06 |

The default prompt wins outright, so `rag.py`'s `prompt_template` is unchanged - no code
change needed, this run just confirms it's the better of the two.
